# This notebook is ment to test the methodes of the Cell class and visualize it using synthetic data


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn

from tqdm import tqdm
from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook
from bokeh.palettes import Spectral10, Colorblind8
from concurrent.futures import ProcessPoolExecutor
from scipy.ndimage import gaussian_filter1d

# Import the cell analysis classes with reload capability
import importlib
import cell_analysis
importlib.reload(cell_analysis)
from cell_analysis import MSNCell, PopulationAnalyzer

print("MSNCell and PopulationAnalyzer classes imported successfully!")

output_notebook()
hv.extension('bokeh')

MSNCell and PopulationAnalyzer classes imported successfully!


Loading BokehJS ...

In [9]:
test_cell_id = 1
test_session = 'test_simple_spike_train'

df = pd.DataFrame.from_records([
# Trial 1: GO Left all ones
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 0,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 260,
        'ssd_number': 1.0,
        'neural_data': np.cumsum(np.arange(1000)),
        'trial_number': 1,
    },
    # Trial 2: GO Right al zeros
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 0,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 360,
        'ssd_number': 1.0,
        'neural_data': [],
        'trial_number': 22,
    },
    # Trial 3: GO Left, more spikes
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 180,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 260,
        'ssd_number': 1.0,
        'neural_data': np.arange(1000),
        'trial_number': 3,
    },
    # Trial 4: GO right, spike at go cue
    {
        'cell_ID': test_cell_id,
        'cell_type': 'MSN',
        'trial_session': test_session,
        'type': 'GO',
        'dir': 180,
        'trial_failed': False,
        'go_cue': 200,
        'stop_cue': np.nan,
        'first_relevant_saccade': 260,
        'ssd_number': 1.0,
        'neural_data': [200],
        'trial_number': 400,
    }
])

test_cell = MSNCell(df)
test_cell.plot_raster(epok=(-200, 500)).opts(height=300)

Cell 1 initialized:
  - Total trials: 4
  - Trial types: ['GO']
  - Directions: [np.int64(0), np.int64(180)]
  - SSD numbers: [np.float64(1.0)]
col_names: [100 101 102 103 104 105]
1 (701,)
['#ffffff', '#E69F00', '#F0E442', '#009E73', '#56B4E9', '#D55E00', '#CC79A7', '#000000']
1
(701,)


:Overlay
   .NdOverlay.I :NdOverlay   [Element]
   .HeatMap.I   :HeatMap   [columns,index]   (value)

In [10]:
def create_synthetic_test_data():
    """
    Create synthetic test data for a single cell with controlled spike patterns.
    
    Test data specifications (all successful trials):
    1. GO trials:
       - go_cue at 200 ms
       - Left (dir=180): spikes at 220 and 240 ms
       - Right (dir=0): spikes at 320 and 340 ms
    
    2. STOP trials:
       - go_cue at 200 ms, stop_cue at 500 ms
       - Left (dir=180): spikes at 250 and 550 ms
       - Right (dir=0): spikes at 320 and 540 ms
    
    3. CONT trials:
       - go_cue at 200 ms, stop_cue at 500 ms
       - Left (dir=180): spikes at 250 and 550 ms
       - Right (dir=0): spikes at 320 and 540 ms
    """
    test_cell_id = 9999
    test_session = 'test_session'
    
    trials = [
        # Trial 1: GO Left
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 180,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': np.nan,
            'first_relevant_saccade': 260,
            'ssd_number': np.nan,
            'neural_data': [220, 240],
        },
        # Trial 2: GO Right
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': np.nan,
            'first_relevant_saccade': 360,
            'ssd_number': np.nan,
            'neural_data': [320, 340],
        },
        # Trial 3: STOP Left
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'STOP',
            'dir': 180,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': np.nan,
            'ssd_number': 1,
            'neural_data': [250, 550],
        },
        # Trial 4: STOP Right
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'STOP',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': np.nan,
            'ssd_number': 1,
            'neural_data': [320, 540],
        },
        # Trial 5: CONT Left
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'CONT',
            'dir': 180,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': 570,
            'ssd_number': 1,
            'neural_data': [250, 550],
        },
        # Trial 6: CONT Right
        {
            'cell_ID': test_cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'CONT',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 200,
            'stop_cue': 500,
            'first_relevant_saccade': 560,
            'ssd_number': 1,
            'neural_data': [320, 540],
        },
    ]
    
    return pd.DataFrame(trials)

test_cell = MSNCell(create_synthetic_test_data())
test_cell.plot_raster(epok=(-200, 500)).opts(height=300)

Cell 9999 initialized:
  - Total trials: 6
  - Trial types: ['CONT', 'GO', 'STOP']
  - Directions: [np.int64(0), np.int64(180)]
  - SSD numbers: [np.float64(1.0)]
col_names: [100 101 102 103 104 105]
1 (701,)
['#ffffff', '#E69F00', '#F0E442', '#009E73', '#56B4E9', '#D55E00', '#CC79A7', '#000000']
1
(701,)


:Overlay
   .NdOverlay.I :NdOverlay   [Element]
   .HeatMap.I   :HeatMap   [columns,index]   (value)